# MICrONS two-photon scans, opened cold

This notebook opens the store using nothing but the store itself: no ingest code and no access to the source listings. Everything below comes from the store's own metadata, and the pixels are read through it from BossDB's TIFFs and DANDI's NWB files. It needs zarr, xarray, icechunk and matplotlib, and an Arraylake login to open the repo.

The store references the 19 two-photon functional scans of the MICrONS mouse, as BossDB published them, with each field's position and the scan's volume times from the DANDI 000402 NWB files. Each scan group also holds the scan's NWB file whole, each field's ROI masks, and the scan's cells matched to the electron-microscopy volume at each CAVE version. The store is the private Arraylake repo `earthmover-biology/MICrONS`; you need an Arraylake login with read access (`al auth login`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import zarr
from arraylake import Client

REPO = "earthmover-biology/MICrONS"

repo = Client().get_repo(REPO)
root = zarr.open_group(repo.readonly_session(tag="v2").store, mode="r")

print(root.attrs["title"])
print(root.attrs["description"])
print()
print("license:", root.attrs["license"])
for citation in root.attrs["citation"]:
    print("cite:", citation)
print("export:", root.attrs["tiff_export"])

## The scans

`two-photon` holds one group per scan, named `<session>-<scan>`. Its attributes summarize every scan. It is a plain group, without OME-Zarr metadata: the OME-Zarr hierarchy starts at the scan groups, each a scene that names its field images.

In [ ]:
scans = root["two-photon"].attrs["scans"]
for name, scan in scans.items():
    print(f"{name:>5}: {scan['fields']} fields, {scan['volumes']:,} volumes at {scan['rate_hz']} Hz")
print(f"{sum(s['fields'] for s in scans.values())} fields in {len(scans)} scans")

## One scan's fields in space

A scan group is an OME-Zarr 0.6 scene. Its coordinate system, `motor`, is the microscope's, in micrometers and seconds, and it places each field image by a translation `(t, z, y, x)`: the first volume's time, the field's depth, and the center of its first pixel. Coplanar fields sit side by side.

In [ ]:
scan = root["two-photon/4-7"]
print(scan.attrs["scan"]["coordinates"])
placement = {t["input"]["path"]: t["translation"] for t in scan.attrs["ome"]["scene"]["coordinateTransformations"]}

fig, ax = plt.subplots(figsize=(6, 6))
for path, (t, z, y, x) in placement.items():
    level = scan[f"{path}/0"]
    pixel = scan[path].attrs["annotations"]["pixel_um"]
    height, width = level.shape[2] * pixel, level.shape[3] * pixel
    ax.add_patch(plt.Rectangle((x - pixel / 2, y - pixel / 2), width, height, fill=False,
                               color=plt.cm.viridis(z / 600)))
    ax.text(x + width / 2, y + height / 2, f"{path}\n{z:g} µm", ha="center", va="center", fontsize=8)
ax.autoscale()
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_xlabel("x (µm), posterior to anterior")
ax.set_ylabel("y (µm), lateral to medial")
ax.set_title("scan 4-7: fields placed by the scene")

## One field

A field image has axes `(t, z, y, x)`, one chunk per frame. The z axis is one plane deep; the scene uses it to place the field at its depth. `volume_times`, beside the fields, holds the exact time of each volume.

In [ ]:
field = scan["field-3"]
level = field["0"]
ms = field.attrs["ome"]["multiscales"][0]
axes = [a["name"] for a in ms["coordinateSystems"][0]["axes"]]
scale = ms["datasets"][0]["coordinateTransformations"][0]["scale"]
print(ms["name"], dict(zip(axes, scale)))
print(f"shape {level.shape} {level.dtype}, chunk {level.chunks}")
print("source:", field.attrs["annotations"]["source_url"])
print("pages:", field.attrs["annotations"]["tiff_pages"])

times = scan["volume_times"][:]
frames = level[20000:20100, 0]
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 6))
left.imshow(frames.mean(axis=0), cmap="gray")
left.set_title(f"mean of volumes 20000-20099 ({times[20000]:.0f}-{times[20099]:.0f} s)")
right.plot(times[20000:20100], frames[:, 200:240, 100:140].mean(axis=(1, 2)))
right.set_xlabel("time (s)")
right.set_title("mean of a 40 x 40 pixel window")

## The two exports

BossDB exported the scans twice. Tag `v1` references the first export, which declared the pixels uint16; tag `v2` references the second, which declares the same bytes int16. The few negative values that motion correction leaves read as values near 65,535 in `v1`.

In [ ]:
first = zarr.open_group(repo.readonly_session(tag="v1").store, mode="r")
old = first["two-photon/4-7/field-3/0"][20000, 0]
new = level[20000, 0]
print("v1:", old.dtype, old.min(), old.max(), "| v2:", new.dtype, new.min(), new.max())
print("same bytes:", np.array_equal(old.view(np.int16), new))
for snapshot in repo.ancestry(tag="v2"):
    print(snapshot.id, snapshot.message[:100])

## What processing made of the scan

Baylor's processing found each cell's outline in each field, an ROI (region of interest), and summed the fluorescence inside it in every volume into a trace. Each scan group holds three things for this, read here at `main`:

- `rois-<n>`: field n's ROI masks, an image `(roi, z, y, x)` in the field's own pixel grid, one chunk per ROI. A mask is a weight per pixel, and masks overlap.
- `nwb`: the scan's DANDI NWB file, whole, in the NWB Zarr convention. Every array is read from DANDI's file; the traces, behavior, stimulus and trial tables are there under their NWB paths.
- `units`: the scan's cells as the CAVE database numbers them, `unit_id`, with the field and ROI each one is.

In [ ]:
latest = zarr.open_group(repo.readonly_session("main").store, mode="r")
scan = latest["two-photon/4-7"]
for part, what in scan.attrs["scan"]["processed"].items():
    print(f"{part}: {what}")

FIELD, ROI = 4, 23  # ROI ids count from 1 within a field
masks = scan[f"rois-{FIELD}/0"]
print(f"\nrois-{FIELD}: shape {masks.shape}, chunk {masks.chunks}")
for key, value in scan[f"rois-{FIELD}"].attrs["annotations"].items():
    print(f"  {key}: {value}")

movie = scan[f"field-{FIELD}/0"][20000:20100, 0].mean(axis=0)
weights = masks[:, 0].max(axis=0)
one = masks[ROI - 1, 0]
fig, (left, middle, right) = plt.subplots(1, 3, figsize=(13, 6))
left.imshow(movie, cmap="gray")
left.set_title(f"field-{FIELD}: mean of volumes 20000-20099")
middle.imshow(weights, cmap="magma")
middle.set_title(f"rois-{FIELD}: largest weight at each pixel")
right.imshow(movie, cmap="gray")
right.contour(one > 0, levels=[0.5], colors="cyan")
right.set_title(f"ROI {ROI} on the field")

## A trace from the NWB file

The NWB file keeps NWB's paths: field n's traces are `processing/ophys/Fluorescence/RoiResponseSeries<n>`. Every array there names its axes, so xarray opens the series as one dataset, the traces on `(time, roi)` and their times on `time`. Column r of the traces is ROI id r + 1.

In [ ]:
import xarray as xr

ophys = f"{scan.path}/nwb/processing/ophys"
series = xr.open_zarr(scan.store, group=f"{ophys}/Fluorescence/RoiResponseSeries{FIELD}",
                      consolidated=False, zarr_format=3)
print(series.attrs["description"], "|", series["data"].dims, series["data"].shape)

table = latest[f"{ophys}/ImageSegmentation/PlaneSegmentation{FIELD}"]
print(f"ROI {ROI}: id {table['id'][ROI - 1]}, classified {table['mask_type'][ROI - 1]}")

trace = series["data"].isel(roi=ROI - 1)
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(series["timestamps"], trace, lw=0.4)
ax.set_xlabel("time (s), the scan's clock")
ax.set_ylabel("fluorescence")
ax.set_title(f"scan 4-7, field {FIELD}, ROI {ROI}: the whole scan")

## From a ROI to an EM cell, at two CAVE versions

The CAVE database names a two-photon cell by `unit_id`, numbered across the whole scan; the NWB file numbers ROIs within each field. `units` joins the two: a field's units follow the earlier fields' in order. Under `units/em/<table>`, each CAVE match table gives every unit's matched EM cell, one array per column, with 0 or −1 where the table has no match.

A cell's EM ID (`pt_root_id`) changes whenever proofreading edits it, so each CAVE version is a commit, tagged `cave-v<version>`. Reading the same unit at two tags shows the match as of each version. The NWB file's own EM columns come from version 343 and are stored as float64, which rounds an 18-digit ID; the CAVE arrays are exact.

In [ ]:
TABLE = "coregistration_manual_v4"
units = scan["units"]
print(units.attrs["description"], "\n")

field_of, roi_of = units["field"][:], units["roi"][:]
unit = int(units["unit_id"][(field_of == FIELD) & (roi_of == ROI)][0])
print(f"field {FIELD}, ROI {ROI} is unit {unit}")
print("the NWB file's v343 EM ID, float64:", f"{table['pt_root_id'][ROI - 1]:.0f}")

matched = {}
for tag in ("cave-v943", "cave-v1412"):
    em = zarr.open_group(repo.readonly_session(tag=tag).store, mode="r")[f"{scan.path}/units/em/{TABLE}"]
    matched[tag] = em["pt_root_id"][:]
    print(f"{tag}: unit {unit} -> EM cell {matched[tag][unit - 1]} at EM point {em['pt_position'][unit - 1]}")

before, after = matched["cave-v943"], matched["cave-v1412"]
changed = np.flatnonzero((before != after) & (before != 0)) + 1
print(f"\n{TABLE}: {len(changed)} of {(before != 0).sum()} matched units of scan 4-7 "
      "have a different EM ID at v1412 than at v943")
example = changed[field_of[changed - 1] == FIELD][0]
print(f"for example unit {example} (field {FIELD}, ROI {roi_of[example - 1]}): "
      f"{before[example - 1]} at v943, {after[example - 1]} at v1412")

diff = repo.diff(from_tag="cave-v943", to_tag="cave-v1412")
print("\narrays both tags hold whose chunks differ, under scan 4-7:")
dropped = tuple(g + "/" for g in diff.deleted_groups)
for path in sorted(p for p in diff.updated_chunks
                   if p.startswith(f"/{scan.path}/") and p not in diff.new_arrays and not p.startswith(dropped)):
    print(" ", path.removeprefix(f"/{scan.path}/"))
print("tables v1412 adds:", sorted({p.split("/")[-2] for p in diff.new_arrays if p.startswith(f"/{scan.path}/")}))
print("tables v1412 drops:", sorted(p.rsplit("/", 1)[1] for p in diff.deleted_groups if p.startswith(f"/{scan.path}/")))